# Module 31 — The Actual Pretraining Run

This is what every module since 01 has been building toward: a real
~125M-parameter GPT trained on the Module 30 corpus. This notebook
combines every Phase 4 technique into one training loop — mixed precision
(22), gradient accumulation (23), AdamW with proper parameter groups (24),
warmup + cosine schedule (25), gradient clipping (26), and checkpointing
(27) — around the exact architecture from Modules 10-17.

**Two distinct things happen in this notebook:**
1. The **real config** below is GPT-2-small's actual published architecture
   (124M params) — this is what Module 31's real run uses, on Colab Pro,
   pointed at Module 30's full 2.5B-token corpus.
2. What's **actually executed and verified in this session** is a
   mechanics smoke test: the real 124M-param model, the real training loop,
   run for a handful of steps on this machine's GPU against synthetic data
   — proving every piece works correctly together, without spending the
   real compute budget on a run that hasn't been sanity-checked yet.

## 1. The real architecture: GPT-2-small's published config

Reusing Modules 10-17's pieces (copied in, not re-explained), plus one
detail those modules glossed over: **GPT-2's actual initialization
scheme**. Module 14 showed that deep residual stacks need their scale
managed carefully; GPT-2's fix is to initialize every weight from
`N(0, 0.02^2)`, and additionally scale each block's two residual-branch
output projections (attention's `Wo`, the feed-forward's second `Linear`)
by an extra `1/sqrt(2 * num_layers)` — since `num_layers` blocks each add
2 residual contributions, this keeps the accumulated residual-stream
variance from growing unboundedly with depth.

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


def scaled_dot_product_attention(Q, K, V, causal=True):
    d_k = Q.shape[-1]
    scores = (Q @ K.transpose(-2, -1)) / math.sqrt(d_k)
    if causal:
        seq_len_q, seq_len_k = scores.shape[-2], scores.shape[-1]
        mask = torch.triu(torch.ones(seq_len_q, seq_len_k, device=scores.device), diagonal=1).bool()
        scores = scores.masked_fill(mask, float("-inf"))
    weights = F.softmax(scores, dim=-1)
    return weights @ V, weights

def split_heads(t, num_heads):
    *batch_dims, seq_len, d_model = t.shape
    d_k = d_model // num_heads
    return t.view(*batch_dims, seq_len, num_heads, d_k).transpose(-3, -2)

def merge_heads(t):
    *batch_dims, num_heads, seq_len, d_k = t.shape
    return t.transpose(-3, -2).contiguous().view(*batch_dims, seq_len, num_heads * d_k)

class MultiHeadAttention(nn.Module):
    def __init__(self, d_model, num_heads):
        super().__init__()
        self.num_heads = num_heads
        self.Wq = nn.Linear(d_model, d_model, bias=False)
        self.Wk = nn.Linear(d_model, d_model, bias=False)
        self.Wv = nn.Linear(d_model, d_model, bias=False)
        self.Wo = nn.Linear(d_model, d_model, bias=False)
    def forward(self, x, causal=True):
        Q, K, V = self.Wq(x), self.Wk(x), self.Wv(x)
        Qh, Kh, Vh = split_heads(Q, self.num_heads), split_heads(K, self.num_heads), split_heads(V, self.num_heads)
        out, _ = scaled_dot_product_attention(Qh, Kh, Vh, causal=causal)
        return self.Wo(merge_heads(out))

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        d_ff = d_ff or 4 * d_model
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model))
    def forward(self, x):
        return self.net(x)

class TransformerBlock(nn.Module):
    def __init__(self, d_model, num_heads, d_ff=None):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = MultiHeadAttention(d_model, num_heads)
        self.ln2 = nn.LayerNorm(d_model)
        self.ffn = FeedForward(d_model, d_ff)
    def forward(self, x):
        x = x + self.attn(self.ln1(x), causal=True)
        x = x + self.ffn(self.ln2(x))
        return x

class GPT(nn.Module):
    def __init__(self, vocab_size, d_model, num_heads, num_layers, max_seq_len, d_ff=None):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_embed = nn.Embedding(vocab_size, d_model)
        self.pos_embed = nn.Embedding(max_seq_len, d_model)
        self.blocks = nn.ModuleList([TransformerBlock(d_model, num_heads, d_ff) for _ in range(num_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, vocab_size, bias=False)
        self.head.weight = self.token_embed.weight
        self.apply(self._init_weights)
        for name, p in self.named_parameters():
            if name.endswith("attn.Wo.weight") or name.endswith("ffn.net.2.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * num_layers))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx):
        seq_len = idx.shape[-1]
        positions = torch.arange(seq_len, device=idx.device)
        x = self.token_embed(idx) + self.pos_embed(positions)
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        return self.head(x)


# GPT-2-small\'s real, published config
VOCAB_SIZE = 50257   # tiktoken "gpt2" vocab size (Module 20) - matches Module 30\'s corpus
D_MODEL = 768
NUM_HEADS = 12
NUM_LAYERS = 12
BLOCK_SIZE = 1024

device = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(42)
model = GPT(VOCAB_SIZE, D_MODEL, NUM_HEADS, NUM_LAYERS, BLOCK_SIZE).to(device)
n_params = sum(p.numel() for p in model.parameters())
print(f"{n_params:,} parameters (GPT-2-small is publicly documented as ~124M)")

## 2. Sanity-checking the initialization fix

Before spending any real compute, confirm initial loss on data with **no**
learnable structure lands near the theoretical floor
(`ln(vocab_size)`) — not wildly higher, which would mean something's
broken in the init or architecture before training even starts.

In [ ]:
import numpy as np

rng = np.random.default_rng(0)
random_tokens = rng.integers(0, VOCAB_SIZE, size=50_000, dtype=np.uint16)

x = torch.from_numpy(random_tokens[:BLOCK_SIZE].astype(np.int64)).unsqueeze(0).to(device)
y = torch.from_numpy(random_tokens[1:BLOCK_SIZE + 1].astype(np.int64)).unsqueeze(0).to(device)

with torch.no_grad():
    logits = model(x)
    initial_loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1)).item()

theoretical_floor = math.log(VOCAB_SIZE)
print(f"initial loss: {initial_loss:.3f}")
print(f"theoretical floor (ln(vocab_size)): {theoretical_floor:.3f}")
assert abs(initial_loss - theoretical_floor) < 1.0, "initialization looks broken - loss should start near the theoretical floor"
print("Confirmed: initialization is properly scaled (Module 14\'s residual-variance lesson, applied for real).")

## 3. Data loading from Module 30's memmap corpus format

Points at Module 30's real output file. For this session's smoke test, a
small synthetic random-token file stands in for it (proving the loading
and training mechanics, not the real corpus - Module 30 already verified
the corpus-building pipeline itself).

In [ ]:
CORPUS_BIN_PATH = "smoke_test_corpus.bin"  # real run: point this at Module 30\'s full output
random_tokens.tofile(CORPUS_BIN_PATH)

def load_corpus(bin_path):
    return np.memmap(bin_path, dtype=np.uint16, mode="r")

def get_batch(data, block_size, batch_size, device):
    ix = np.random.randint(0, len(data) - block_size - 1, size=batch_size)
    x = torch.stack([torch.from_numpy(data[i:i + block_size].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i + 1:i + 1 + block_size].astype(np.int64)) for i in ix])
    return x.to(device), y.to(device)


corpus = load_corpus(CORPUS_BIN_PATH)
print(f"loaded corpus: {len(corpus):,} tokens (memmap, not fully in RAM)")

## 4. The optimizer: decay/no-decay parameter groups

A real detail this project's earlier modules glossed over: weight decay
(Module 24) should only apply to weight *matrices*, not to biases or
LayerNorm gain/bias parameters (1-dimensional parameters) — decaying those
doesn't correspond to the same "shrink large weights" intuition and
empirically hurts. `nanoGPT` and GPT-2-style training configs always split
parameters this way.

In [ ]:
decay_params = [p for p in model.parameters() if p.dim() >= 2]
no_decay_params = [p for p in model.parameters() if p.dim() < 2]
print(f"decay params:    {sum(p.numel() for p in decay_params):,}")
print(f"no-decay params: {sum(p.numel() for p in no_decay_params):,}")

WEIGHT_DECAY = 0.1
PEAK_LR = 6e-4

optimizer = torch.optim.AdamW(
    [
        {"params": decay_params, "weight_decay": WEIGHT_DECAY},
        {"params": no_decay_params, "weight_decay": 0.0},
    ],
    lr=PEAK_LR, betas=(0.9, 0.95),
)

## 5. The full training loop

Every Phase 4 technique, combined: gradient accumulation (23) around
autocast bf16 (22) micro-steps, gradient clipping (26) before the
optimizer step, warmup + cosine schedule (25) each step, and periodic
checkpointing (27, using the exact save/load functions from that module).

In [ ]:
def lr_multiplier(step, warmup_steps, total_steps):
    if step < warmup_steps:
        return step / max(1, warmup_steps)
    progress = min((step - warmup_steps) / max(1, total_steps - warmup_steps), 1.0)
    return 0.5 * (1.0 + math.cos(math.pi * progress))


def save_checkpoint(model, optimizer, step, path):
    torch.save({
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "step": step,
    }, path)


MICRO_BATCH_SIZE = 4       # real run: as large as GPU memory allows
GRAD_ACCUM_STEPS = 4       # real run: tuned so micro_batch * grad_accum * block_size hits the target effective batch
MAX_GRAD_NORM = 1.0
WARMUP_STEPS = 5           # real run: ~a few hundred steps
TOTAL_STEPS = 10           # real run: total_tokens / (effective_batch_size * block_size) - see Module 30
CHECKPOINT_EVERY = 5
CHECKPOINT_PATH = "smoke_test_checkpoint.pt"

losses = []
for step in range(TOTAL_STEPS):
    lr = PEAK_LR * lr_multiplier(step, WARMUP_STEPS, TOTAL_STEPS)
    for group in optimizer.param_groups:
        group["lr"] = lr

    optimizer.zero_grad()
    for micro_step in range(GRAD_ACCUM_STEPS):
        x, y = get_batch(corpus, BLOCK_SIZE, MICRO_BATCH_SIZE, device)
        with torch.autocast(device_type="cuda" if device == "cuda" else "cpu", dtype=torch.bfloat16):
            logits = model(x)
            loss = F.cross_entropy(logits.view(-1, VOCAB_SIZE), y.view(-1)) / GRAD_ACCUM_STEPS
        loss.backward()

    torch.nn.utils.clip_grad_norm_(model.parameters(), MAX_GRAD_NORM)
    optimizer.step()

    step_loss = loss.item() * GRAD_ACCUM_STEPS
    losses.append(step_loss)
    print(f"step {step:2d}   lr {lr:.2e}   loss {step_loss:.3f}")

    if (step + 1) % CHECKPOINT_EVERY == 0:
        save_checkpoint(model, optimizer, step, CHECKPOINT_PATH)
        print(f"  (checkpoint saved at step {step})")

print("\nSmoke test complete - every Phase 4 technique ran together without error.")

## 6. Verifying the checkpoint actually saved something usable

In [ ]:
checkpoint = torch.load(CHECKPOINT_PATH, weights_only=True)
fresh_model = GPT(VOCAB_SIZE, D_MODEL, NUM_HEADS, NUM_LAYERS, BLOCK_SIZE).to(device)
fresh_model.load_state_dict(checkpoint["model"])
loaded_step = checkpoint["step"]
print(f"Checkpoint loaded successfully from step {loaded_step} into a brand-new model instance.")

## Recap, and how to run this for real

This session verified: the real 124M-parameter GPT-2-small architecture,
proper initialization (loss starts at the theoretical floor on unstructured
data), memmap-based data loading, decay/no-decay AdamW groups, gradient
accumulation + mixed precision, gradient clipping, a warmup/cosine
schedule, and checkpointing — all running together correctly on this
machine's GPU.

**To run the real pretrain on Colab Pro:**
1. Run Module 30 with `TARGET_TOTAL_TOKENS = 2_500_000_000` (no page cap)
   to build the real corpus, and upload the resulting `.bin` to Drive.
2. Point `CORPUS_BIN_PATH` at that file.
3. Set `TOTAL_STEPS` so that
   `TOTAL_STEPS * GRAD_ACCUM_STEPS * MICRO_BATCH_SIZE * BLOCK_SIZE ≈ 2.5B`
   tokens, `WARMUP_STEPS` to a few hundred, and tune `MICRO_BATCH_SIZE` up
   to whatever the A100's memory allows.
4. Mount Drive, save `CHECKPOINT_PATH` there, and checkpoint often enough
   (Module 27) that a Colab disconnect doesn't cost much progress.

Module 32 covers instruction fine-tuning — turning this raw pretrained
model into something that follows conversational turns, once a real
checkpoint exists to fine-tune.